# Dynamic Programming

DP is a technique used for solving problems by breaking them into smaller subproblems. The main thing is that many problems have overlapping subproblems, and we don't want to end up solving the same smaller problem multiples times. What DP does is that it solves each subproblem once and saves the result so it can solve the rest.

Every DP algorithm has two steps:
1. Define subproblems: what smaller version of the problem do you need to solve?
2. Find the dependency relation: how does the original problem depend on the subproblems?

And the framework is:

```
base case
for loop
    dependency relation
return Goal
```

### Fibonacci

Fibonacci is a sequence where: f(0) = 0, f(1) = 1, f(n) = f(n-1) + f(n-2)

The recursive approach:

```
f(n):
    if n = 0:
        return 0
    if n = 1:
        return 1
    return f(n-1) + f(n-2)
```

This is actually super slow. Why? If we look at the call tree for f(6), f(4) gets computed twice, f(3) gets computes three times. We're doing the same thing over and over again.

If we use DP instead, we can just save the results in a table and use them to compute the rest:

```
f[0] = 0
f[1] = 1

for v = 2 to n:
    f[v] = f[v-1] + f[v-2]
return f[n]
```

The time for this is $O(n)$. If we see how the steps are applied:

**Subproblems**: f(i) for 0 <= i <= n

**Dependency relation**: f(i) = f(i-1) + f(i-2)

**Base case**: f(0)=0, f(1)=1

**Goal**: f(n)

The tanle is O(n), but do we actually need the whole table? At any point we only need the previous two values, so we could do O(1) space:

```
a = 0
b = 1
for i = 2 to n:
    c = a + b
    a = b
    b = c
return c

## Finding the Largest Number

Input: an array A[1..n]
Output: the largest number in A

We already know we can do:

```
x = A[1]
for i = 2 to n:
    if x < A[i]:
        x = A[i]
return x
```

Time: O(n)

If we apply DP:

**Subproblems**: f(i) = the largest number in A[1...i], for i <= i <= n

**Goal**: f(n)

**Dependency relation**: f(i) = max(A[i], f(i-1)) *the largest in the first i elements is either the new element A[i] or whatever was already the largest in the first i-1

**Base case**: f(1) = A[1]

```
f[1] = A[1]
for i = 2 to n:
    x = max(x, A[i])
return x
```

## Knapsack Problem

Input: n items of sizes a1, a2, ... an, and a knapsack of size M. All values are positive integers

Output: find a subset of items whose total sum equals exactly  M

We can simplify the problem first: just answer yes or no, does any subset sum to M?

We can brute force, try all combinations, but that's $2^n$ possibilities.

So, instead, we apply DP. The key question though is, how do we reduce from size n to n-1? We can look at an item $a_n$ and ask if it's in the solution or not.

**Case 1**: If $a_n$ is NOT in the solution, then the answer comes entirely from the first n-1 items summing to M, so we need to solve f(n-1, M)

**Case 2**: If $a_n$ is in the solution, then the remaining items from the first n-1 must sum to M - $a_n$, so we need to solve f(n-1, M - $a_n$)

If either case gives yes, the answer is yes. So:

```f(n, M) = max{ f(n-1, M), f(n-1, M - $a_n$) }```

**Subproblems**: f(i, k) for 0 <= i <= n, 0 <= k <= M (does a subset of the first i items sum to exactly k?)

**Goal**: f(n, M)

**Dependency relation**: f(i, k) = max{ f(i-1, k), f(i-1, k - $a_i$) }

**Base cases**:
- f(i, 0) = 1 for all i
- f(0, k) = 0 for i <= k <= M

```
for k = 1 to M:
    f[0, k] = 0
for i = to n:
    f[i, 0] = 1

for i = 1 to n:
    for k = 1 to M:
        if k >= $a_i$:
            f[i, k] = max { f[i-1, k], f[i-1, k - $a_i] }
        else:
            f[i, k] = f[i-1, k]

return f[n, M]
```

The time is O(nM), and the space as well, but it can be improved to O(M) by only keeping the previous row.

```
A[0] = 1;  A[k] = 0 for 1 ≤ k ≤ M;
for v = 1 to n:
  for k = M downto 1:
    A[k] = max{ A[k], A[k - a_v] };
return A[M];
```

**Backtracking:** Once we have the table, we can walk backwards through the table to find which items are in the solution

```
k = M;  i = n;
while k > 0:
  if f(i-1, k-a_i) = 1:
    report a_i; 
    k = k - a_i;
  i--;
```

## LCS

Input: Two sequences

X = x1, x2... xn

Y = y1, y2... ym

Output: LCS(X, Y), the longest subsequence appearing in both X and Y

A subsequence of X is a subset following the same order as in X, but not neccessarily consecutive.

A simplified version of the problem is to compute the length of LCS(X, Y). If we do it by brute-force, there are $2^n$ combinations to look at.

We can use DP to reduce the problem size from (n,m) to (n-1,m). Let's say we don't know the LCS yet, but we know it exists. We can call it Z, so Z = z1, z2... zk. Now, the very last chararacter of Z is zk, and the last characters of X and Y are xn and ym. There are three possible situations.

**Case 1. zk is not xn:** This means the last item of X is NOT part of the LCS, so we can safely ignore xn entirely and look for the LCS of x1...xn-1 and Y, so the problem is now size (n-1,m).

**Case 2. zk is not xm:** Similar to above but for Y. Now the size is (n, m-1).

**Case 3. zk = xn = ym:** If xn = ym and xn matches both of them. The size it's definitely (n-1,m-1).

Now we get to apply DP.

**Subproblems**: f(i,j) = the length of the LCS of the first i characters of X, and the first j characters of Y
**Depedency relations**:
```
if xi = yj:
    f(i, j) = f(i-1, j-1) + 1

if xi != yj:
    f(i, j) = max{ f(i-1, j), f(i, j-1)}
```
**Goal**: f(n,m)
**Base case**: i=0 or j=0, return 0

## Maximum Subarray

**Input**: An array A[1...n]

**Output**: A non-empty subarray A[i...j], whose total sum is the largest possible

For example:
```A = [ 4, 3, -5, 6 0, 1, 10, -2, 5, -1, 6, -7, 2]
The answer is A[9...11] = [5, -1, 6]

The simplified version would be to compute the sum of the max subarray.

Now, for this, we can't simply reduce the prbolem size from n to n-1, because the optimal subarray might be fully from 1..n-1 or end at n. But every subarray does have to end somewhere, so it can be at 1 <= j <= n.

**Subproblem**: f(j) (the sum of the maximum subarray ending at index j)
**Goal:** max{ f(1), f(2)... f(n) }
**Dependency relation**: f(j) = A[j] + max { 0, f(j-1) }
**Base case**: f(1) = A[1]

```
f[1] = A[1]
for j = 2 to n
    f[j] = A[j] + max{ 0, f[j-1] }

return max{ f[1], f[2], …, f[n] }
```

We can make the space O(1):

```
a = A[1];   opt = a;
for j = 2 to n
    a = A[j] + max{ 0, a }
    if opt < a
        opt = a
return opt
```